# Kandinsky Game→Audio — Architecture Step 2: Game-to-Audio Branch

This notebook prototypes the **shape and residual interface** for adding game-event context to Kandinsky's audio stream.

**What it does:** inspects the pinned Kandinsky source and runs a tiny PyTorch demonstration of audio queries attending to game-event tokens.

**What it does not do:** patch Kandinsky, load model weights, train, or generate audio. The demo tokens and parameters are random; they do not demonstrate learned game control.

## The proposed conditioning layout

- Text → audio: existing Kandinsky path.
- Video ↔ audio: existing Kandinsky path when both modalities are present.
- Game events → audio: the new path prototyped here.

The target is **not** all-to-all attention. Game events are control context for sound, so this first design adds only a Game→Audio branch. It must run both when audio is generated alone and when audio is generated jointly with video.

In the joint path, the new branch is placed after the existing text and optional video/audio updates, before the audio feed-forward block. In the audio-only path, it follows the existing text update.

In [ ]:
import subprocess
from pathlib import Path
import torch
from torch import nn

print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("No checkpoint download, training, or audio generation in this notebook.")

## 1. Inspect the actual insertion points

We fetch the Kandinsky source at the same pinned revision as Step 1. This downloads code only, not model weights.

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-arch-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()

if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True
    )

dit_path = K6_DIR / "kandinsky/core/components/dit.py"
lines = dit_path.read_text(encoding="utf-8").splitlines()
print("Pinned source commit:", subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip())

anchors = [
    "class MultiheadCrossAttention",
    "class FusedTransformerDecoderBlock",
    "# ---- audio backbone ----",
    "# ---- cross-modal attention ----",
    "# ---- FFN ----",
]
for anchor in anchors:
    hits = [i for i, line in enumerate(lines) if anchor in line]
    print(f"\n--- {anchor} ---")
    for i in hits[:1]:
        lo, hi = max(0, i - 2), min(len(lines), i + 9)
        print("\n".join(f"{j + 1}: {lines[j]}" for j in range(lo, hi)))

## 2. Define the adapter interface

The audio hidden states supply the queries. Game tokens supply keys and values. Their feature dimensions can differ: the cross-attention projects them into the audio attention space.

Shapes used below:

- Audio hidden states: `[batch, audio_tokens, audio_dim]`
- Game tokens: `[batch, game_tokens, game_dim]`
- Game mask: `[batch, game_tokens]`; `True` means a valid token.

The zero-initialized residual gate makes the untrained branch leave the audio stream unchanged. This tiny demo uses a scalar gate for clarity. A real Kandinsky integration should use a gate consistent with its timestep-conditioned modulation.

In [ ]:
class GameToAudioBranch(nn.Module):
    """Small shape prototype: audio queries attend to masked game tokens."""

    def __init__(self, audio_dim, game_dim, num_heads):
        super().__init__()
        if audio_dim % num_heads:
            raise ValueError("audio_dim must be divisible by num_heads")

        self.audio_norm = nn.LayerNorm(audio_dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=audio_dim,
            num_heads=num_heads,
            kdim=game_dim,
            vdim=game_dim,
            batch_first=True,
        )
        # At initialization, this branch contributes exactly zero.
        self.gate = nn.Parameter(torch.zeros(()))

    def forward(self, audio_hidden, game_tokens, game_token_mask):
        if audio_hidden.ndim != 3 or game_tokens.ndim != 3:
            raise ValueError("Expected audio and game tensors with shape [batch, tokens, dim]")
        if audio_hidden.shape[0] != game_tokens.shape[0]:
            raise ValueError("Audio and game tensors must have the same batch size")
        if game_token_mask.shape != game_tokens.shape[:2]:
            raise ValueError("game_token_mask must match [batch, game_tokens]")

        valid = game_token_mask.to(device=game_tokens.device, dtype=torch.bool)
        if not valid.any(dim=1).all():
            raise ValueError("Each example must contain at least one valid game token")

        update, weights = self.cross_attention(
            query=self.audio_norm(audio_hidden),
            key=game_tokens,
            value=game_tokens,
            key_padding_mask=~valid,  # PyTorch masks True positions out.
            need_weights=True,
            average_attn_weights=False,
        )
        output = audio_hidden + torch.tanh(self.gate) * update
        return output, weights

## 3. Run a tiny shape demonstration

These dimensions are deliberately small and do not match a Kandinsky checkpoint. The demonstration checks three interface properties:

1. The branch returns one updated vector per audio token.
2. With the gate at zero, the audio representation is unchanged.
3. Masked game tokens do not affect the attention output.

Passing these checks validates tensor plumbing only; it says nothing about sound quality or learned control.

In [ ]:
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 2
audio_length = 24
game_length = 8
audio_dim = 128
game_dim = 64
num_heads = 4

audio_hidden = torch.randn(batch_size, audio_length, audio_dim, device=device)
game_tokens = torch.randn(batch_size, game_length, game_dim, device=device)
game_token_mask = torch.tensor([
    [True, True, True, True, False, False, False, False],
    [True, True, True, True, True, True, False, False],
], device=device)

branch = GameToAudioBranch(audio_dim, game_dim, num_heads).to(device)

with torch.no_grad():
    unchanged, _ = branch(audio_hidden, game_tokens, game_token_mask)
    assert torch.equal(unchanged, audio_hidden), "Zero gate must preserve the input exactly."

    branch.gate.fill_(0.25)
    output_a, attention_weights = branch(audio_hidden, game_tokens, game_token_mask)

    changed_masked_tokens = game_tokens.clone()
    changed_masked_tokens[~game_token_mask] += 1000.0
    output_b, _ = branch(audio_hidden, changed_masked_tokens, game_token_mask)
    assert torch.allclose(output_a, output_b, atol=1e-6), "Masked tokens must have no effect."

print("Audio input:", tuple(audio_hidden.shape))
print("Game input:", tuple(game_tokens.shape))
print("Output:", tuple(output_a.shape))
print("Attention weights:", tuple(attention_weights.shape))
print("Zero-gate preservation: passed")
print("Masked-token isolation: passed")
print("Non-zero gated residual magnitude:", (output_a - audio_hidden).abs().mean().item())

## 4. How the same branch fits both Kandinsky paths

| Generation path | Existing audio context before the new branch | Game branch |
|---|---|---|
| Audio only | Audio self-attention, then text cross-attention | Game→Audio, then audio FFN |
| Video + audio | Audio self-attention and text cross-attention, then existing video/audio exchange | Game→Audio, then audio FFN |

The same game branch can be called in both cases because it consumes audio hidden states and game tokens; it does not require video tokens. Event-time information must be present in the game tokens, otherwise global attention has no explicit basis for placing an event at the right point in the sound.

## What this step establishes — and what remains

**Established:** a separate Game→Audio cross-attention branch can accept game tokens with a different feature size, ignore masked fields, and preserve the original audio state when its residual gate is closed.

**Not established:** integration into Kandinsky's real block, checkpoint compatibility, sound quality, or control over material/intensity/timing. No paired game-event/audio data has been used, and nothing has been trained.

**Next architecture step:** patch a local copy of the pinned Kandinsky block so the branch runs in both audio-only and video+audio paths, while leaving the pretrained text and video/audio branches intact. Then inspect checkpoint loading and missing new parameters before choosing a fine-tuning method.